# Regression Model (Ratings Prediction)

This notebook implements a regression-based recommender that predicts user star ratings for restaurants.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import ast
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder


## 1) Load Data

In [ ]:
# Update DATA_DIR if your dataset is stored somewhere else.
DATA_DIR = r"C:\Users\TNath\OneDrive\School\2025-2026\Spring\Recommender Systems\Recommender Systems Code\Final Project\team_project_dataset"

restaurants_path = f"{DATA_DIR}\restaurants_santa_barbara.csv"
train_reviews_path = f"{DATA_DIR}\train_reviews_santa_barbara.csv"
test_reviews_path = f"{DATA_DIR}\test_reviews_santa_barbara.csv"

df_restaurants = pd.read_csv(restaurants_path)
df_train_reviews = pd.read_csv(train_reviews_path)
df_test_reviews = pd.read_csv(test_reviews_path)

print("Restaurants:", df_restaurants.shape)
print("Train reviews:", df_train_reviews.shape)
print("Test reviews:", df_test_reviews.shape)


## 2) Feature Preparation

In [ ]:
def count_categories(cat_string):
    if pd.isna(cat_string):
        return 0
    return len([c.strip() for c in str(cat_string).split(',') if c.strip()])


def count_attributes(attr_string):
    if pd.isna(attr_string):
        return 0
    try:
        parsed = ast.literal_eval(attr_string)
        return len(parsed) if isinstance(parsed, dict) else 0
    except Exception:
        return 0

restaurant_features = df_restaurants.copy()
restaurant_features["num_categories"] = restaurant_features["categories"].apply(count_categories)
restaurant_features["num_attributes"] = restaurant_features["attributes"].apply(count_attributes)

restaurant_cols = [
    "business_id",
    "name",
    "city",
    "state",
    "is_open",
    "review_count",
    "latitude",
    "longitude",
    "stars",
    "num_categories",
    "num_attributes",
]

restaurant_features = restaurant_features[restaurant_cols].rename(
    columns={"stars": "business_avg_stars", "review_count": "business_review_count"}
)

train_df = df_train_reviews.merge(restaurant_features, on="business_id", how="left")
test_df = df_test_reviews.merge(restaurant_features, on="business_id", how="left")

train_df = train_df.dropna(subset=["stars"]).copy()
test_df = test_df.dropna(subset=["stars"]).copy()

print("Train rows used:", train_df.shape[0])
print("Test rows used:", test_df.shape[0])


## 3) Baseline (Global Mean)

In [ ]:
y_train = train_df["stars"].astype(float)
y_test = test_df["stars"].astype(float)

global_mean = y_train.mean()
baseline_pred = np.full(shape=len(y_test), fill_value=global_mean)

baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"Global mean baseline: {global_mean:.4f}")
print(f"Baseline RMSE: {baseline_rmse:.4f}")
print(f"Baseline MAE:  {baseline_mae:.4f}")


## 4) Regression Pipeline

In [ ]:
feature_cols = [
    "user_id",
    "business_id",
    "text",
    "city",
    "state",
    "name",
    "is_open",
    "business_review_count",
    "latitude",
    "longitude",
    "business_avg_stars",
    "num_categories",
    "num_attributes",
]

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

numeric_features = [
    "is_open",
    "business_review_count",
    "latitude",
    "longitude",
    "business_avg_stars",
    "num_categories",
    "num_attributes",
]

categorical_features = [
    "user_id",
    "business_id",
    "city",
    "state",
    "name",
]

text_feature = "text"

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5)),
    ]
)

text_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="")),
        ("flatten", FunctionTransformer(lambda x: x.squeeze(), validate=False)),
        ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=3)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        ("txt", text_transformer, [text_feature]),
    ],
    remainder="drop",
)

regressor = Ridge(alpha=2.0, random_state=42)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", regressor),
    ]
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

y_pred = np.clip(y_pred, 1.0, 5.0)


## 5) Evaluation

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Regression model performance")
print("-" * 40)
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R^2:  {r2:.4f}")

print("\nComparison vs baseline")
print("-" * 40)
print(f"Baseline RMSE: {baseline_rmse:.4f}")
print(f"Model RMSE:    {rmse:.4f}")
print(f"Improvement:   {baseline_rmse - rmse:.4f}")


## 6) Save Predictions

In [ ]:
predictions = test_df[["user_id", "business_id", "stars"]].copy()
predictions = predictions.rename(columns={"stars": "actual_stars"})
predictions["predicted_stars"] = y_pred
predictions["abs_error"] = (predictions["actual_stars"] - predictions["predicted_stars"]).abs()

predictions.to_csv("regression_predictions.csv", index=False)
predictions.head(10)


## 7) Example: Top Predicted Restaurants for a User (from test set candidates)

In [ ]:
example_user = predictions["user_id"].iloc[0]

user_recs = (
    predictions[predictions["user_id"] == example_user]
    .sort_values("predicted_stars", ascending=False)
    .head(10)
    .merge(df_restaurants[["business_id", "name", "categories"]], on="business_id", how="left")
)

print(f"Example user_id: {example_user}")
user_recs[["business_id", "name", "predicted_stars", "actual_stars", "categories"]]
